# Model Benchmarking

This notebook documents the forecasting-model comparison developed for the
Campania Financial & ESG Forecasting project.

The original company-level dataset is proprietary and cannot be redistributed.
Accordingly, this public notebook separates:

1. the **original experimental design and reported benchmark results**; and
2. a **small synthetic reproducibility demo** showing the modelling workflow.

The benchmark compares:

- Linear Regression;
- Random Forest;
- XGBoost;
- Prophet;
- Long Short-Term Memory (LSTM).

The comparison target used for model selection was **EBITDA 2024**, predicted
from historical EBITDA observations for 2015–2023.

## 1. Original experimental design

The original project used:

- a **75/25 train-test split**;
- stratification by **company size**;
- **Mean Squared Error (MSE)** as the benchmark metric;
- hyperparameter search for the tree-based models;
- the same held-out test logic for model comparison.

The purpose of the benchmark was to identify a model that combined predictive
accuracy with manageable model complexity before extending the analysis to
the final financial and ESG targets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

## 2. Reported benchmark results

The table below reproduces the final MSE values reported in the research paper.
These values come from the original project dataset, **not** from the synthetic
demonstration used later in this notebook.

In [ ]:
benchmark_results = pd.DataFrame(
    {
        "Model": [
            "Linear Regression",
            "Random Forest",
            "XGBoost",
            "Prophet",
            "LSTM",
        ],
        "MSE": [
            0.00299,
            0.00285,
            0.00292,
            0.00392,
            0.00287,
        ],
    }
).sort_values("MSE")

benchmark_results

In [ ]:
ax = benchmark_results.plot(
    x="Model",
    y="MSE",
    kind="bar",
    legend=False,
    figsize=(9, 4),
)

ax.set_title("Model Benchmark — Reported Project Results")
ax.set_ylabel("Mean Squared Error")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Model selection

**Random Forest achieved the lowest reported MSE (0.00285)**, closely followed
by LSTM (0.00287).

Random Forest was selected for the subsequent forecasting analysis because it
provided the lowest benchmark error while remaining less complex and easier to
operate than the recurrent neural-network alternative.

## 3. Selected tree-model configurations

The final selected configurations reported for the two tuned tree-based models
were:

**Random Forest**
- `n_estimators = 100`
- `max_depth = 10`
- `min_samples_leaf = 10`

**XGBoost**
- `learning_rate = 0.01`
- `max_depth = 10`
- `n_estimators = 500`
- `min_child_weight = 1`

The full project evaluated multiple parameter combinations before choosing the
configuration minimizing MSE.

## 4. Public reproducibility demo

To demonstrate the modelling workflow without exposing proprietary records, we
generate a small synthetic panel of company-level EBITDA histories.

The synthetic results below are **illustrative only** and must not be interpreted
as the results of the original Campania dataset.

In [ ]:
rng = np.random.default_rng(42)

n_companies = 300
years = list(range(2015, 2025))

company_size = rng.choice(
    ["micro", "small", "medium/large"],
    size=n_companies,
    p=[0.35, 0.45, 0.20],
)

base = rng.lognormal(mean=1.4, sigma=0.55, size=n_companies)
growth = rng.normal(loc=0.045, scale=0.035, size=n_companies)

synthetic = pd.DataFrame(
    {
        "company_id": [f"COMP_{i:04d}" for i in range(n_companies)],
        "company_size": company_size,
    }
)

for step, year in enumerate(years):
    noise = rng.normal(loc=0.0, scale=0.12, size=n_companies)
    synthetic[f"ebitda_{year}"] = (
        base * (1 + growth) ** step + noise
    ).clip(min=0)

synthetic.head()

In [ ]:
feature_columns = [f"ebitda_{year}" for year in range(2015, 2024)]

X = synthetic[feature_columns]
y = synthetic["ebitda_2024"]
strata = synthetic["company_size"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=strata,
)

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))

## 5. Linear Regression

Linear Regression provides a transparent baseline and estimates 2024 EBITDA as
a linear combination of historical annual observations.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)
lr_demo_mse = mean_squared_error(y_test, lr_predictions)

print(f"Synthetic-demo MSE — Linear Regression: {lr_demo_mse:.6f}")

## 6. Random Forest

Random Forest combines multiple decision trees and can capture non-linear
relationships without requiring an explicit functional form.

The public demo uses the final configuration selected in the original project.

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
rf_demo_mse = mean_squared_error(y_test, rf_predictions)

print(f"Synthetic-demo MSE — Random Forest: {rf_demo_mse:.6f}")

## 7. XGBoost

The original project also benchmarked gradient boosting through XGBoost.
The selected parameter configuration is reproduced below.

The cell runs only when the `xgboost` package is available.

In [ ]:
try:
    import xgboost as xgb

    xgb_model = xgb.XGBRegressor(
        learning_rate=0.01,
        max_depth=10,
        n_estimators=500,
        min_child_weight=1,
        random_state=42,
        n_jobs=-1,
    )

    xgb_model.fit(X_train, y_train)

    xgb_predictions = xgb_model.predict(X_test)
    xgb_demo_mse = mean_squared_error(y_test, xgb_predictions)

    print(f"Synthetic-demo MSE — XGBoost: {xgb_demo_mse:.6f}")

except ImportError:
    print("XGBoost is not installed. Install project requirements to run this cell.")

## 8. Prophet and LSTM

The original benchmark also included two explicitly temporal approaches:

### Prophet
A separate annual time series was modelled for each company. In this setting,
Prophet performed worse than the other approaches. The short annual histories
provide limited seasonal structure, which reduces the advantages of an
additive trend/seasonality model.

### LSTM
An LSTM recurrent neural network was used to model the 2015–2023 EBITDA
sequence and predict EBITDA 2024. Its benchmark performance was very close to
Random Forest, but at higher modelling and training complexity.

The complete project dependencies include both `prophet` and `tensorflow`.
They are intentionally not trained in this lightweight public demo so that the
notebook remains fast to inspect and reproduce.

## 9. Synthetic-demo comparison

The following comparison contains **only metrics generated from the synthetic
data in this notebook**. It is kept separate from the original benchmark table
to avoid mixing illustrative and empirical project results.

In [ ]:
demo_results = [
    {"Model": "Linear Regression", "Synthetic Demo MSE": lr_demo_mse},
    {"Model": "Random Forest", "Synthetic Demo MSE": rf_demo_mse},
]

if "xgb_demo_mse" in globals():
    demo_results.append(
        {"Model": "XGBoost", "Synthetic Demo MSE": xgb_demo_mse}
    )

pd.DataFrame(demo_results).sort_values("Synthetic Demo MSE")

## 10. Why Random Forest was retained

The final project continued with Random Forest because:

- it achieved the lowest reported benchmark MSE;
- it captured non-linear patterns in the company histories;
- its performance was essentially comparable to the LSTM benchmark;
- it required less modelling and training complexity than the deep-learning
  alternative;
- it could be applied consistently to multiple financial and ESG targets.

The selected model was subsequently used to forecast:

- Sales Revenue;
- EBITDA;
- Net Income;
- ESG performance.

## Next Step

The next notebook applies the selected forecasting framework to the final
financial and ESG targets and prepares the predictions used in the downstream
business analysis.